# ETL — VISp Excitatory WNM: Cell-to-Cluster Mapping

Registers MET-type assignments for VISp excitatory whole-neuron morphology cells (`project_id="visp_wnm"`, `dataset_id="visp_exc_wnm"`) as a single `CellToClusterMapping` set against the existing VISp MET-types taxonomy (`hierarchy_id="visp_met_types_taxonomy"`).

This is a *mapping*, not membership: WNM cells did not define the MET-types taxonomy (Patch-seq cells did, in `etl_visp_exc_patchseq_03_…`). Labels were produced by a routed random forest classifier trained on the Patch-seq mMET cohort, sourced from the `predicted_met_type` column of `FullMorphMetaData_Master.csv` with the per-call `probability` recorded on the leaf row only and left null on ancestors (matches the convention in `etl_visp_exc_patchseq_03_…`).

**Prerequisites:** `etl_wnm_exc_01_dataset_dataitem.ipynb`, `etl_visp_met_types_01_cluster.ipynb`.


In [1]:
import pandas as pd
import polars as pl
from deltalake import write_deltalake

from connects_common_connectivity.arrow_utils import (
    attach_linkml_metadata,
    build_arrow_schema,
    models_to_table,
)
from connects_common_connectivity.models import (
    CellToClusterMapping,
    MappingSet,
)
from connects_common_connectivity.write_utils import walk_ancestors


In [2]:
INPUT_CSV                 = "/data/visp-features-and-mapping/FullMorphMetaData_Master.csv"
OUTPUT_ROOT               = "../scratch/em_patchseq_wnm_v1/"

PROJECT_ID                = "visp_wnm"
DATASET_ID                = "visp_exc_wnm"
METTYPE_HIERARCHY_ID      = "visp_met_types_taxonomy"
MAPPING_SET_ID            = "visp_exc_wnm_mettype_mapping"
# Cohort that defined the MET-types and trained the classifier.
MAPPING_SOURCE_DATASET_ID = "visp_exc_patchseq"

print(f"INPUT_CSV                 : {INPUT_CSV}")
print(f"OUTPUT_ROOT               : {OUTPUT_ROOT}")
print(f"PROJECT_ID                : {PROJECT_ID}")
print(f"DATASET_ID                : {DATASET_ID}")
print(f"METTYPE_HIERARCHY_ID      : {METTYPE_HIERARCHY_ID}")
print(f"MAPPING_SET_ID            : {MAPPING_SET_ID}")
print(f"MAPPING_SOURCE_DATASET_ID : {MAPPING_SOURCE_DATASET_ID}")


INPUT_CSV                 : /data/visp-features-and-mapping/FullMorphMetaData_Master.csv
OUTPUT_ROOT               : ../scratch/em_patchseq_wnm_v1/
PROJECT_ID                : visp_wnm
DATASET_ID                : visp_exc_wnm
METTYPE_HIERARCHY_ID      : visp_met_types_taxonomy
MAPPING_SET_ID            : visp_exc_wnm_mettype_mapping
MAPPING_SOURCE_DATASET_ID : visp_exc_patchseq


## Prerequisite check

Read DataItems for this dataset and the MET-types cluster table. Build `{id: parent}` for ancestor walks.

In [3]:
assoc = (
    pl.read_delta(OUTPUT_ROOT + "dataitem_dataset_association/")
      .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("dataset_id") == DATASET_ID))
)
assert assoc.shape[0] > 0, (
    f"etl_wnm_exc_01 must run first — no association rows for dataset_id='{DATASET_ID}'"
)
registered_ids = set(assoc["dataitem_id"].to_list())
print(f"Registered DataItems for {DATASET_ID}: {len(registered_ids)}")

met_clu = (
    pl.read_delta(OUTPUT_ROOT + "cluster/")
      .filter(pl.col("hierarchy_id") == METTYPE_HIERARCHY_ID)
)
assert met_clu.shape[0] > 0, (
    f"etl_visp_met_types_01_cluster must run first — no clusters for {METTYPE_HIERARCHY_ID}"
)
met_parent = dict(zip(met_clu["id"].to_list(), met_clu["parent"].to_list()))
print(f"Clusters loaded: {METTYPE_HIERARCHY_ID}={len(met_parent)}")


Registered DataItems for visp_exc_wnm: 345
Clusters loaded: visp_met_types_taxonomy=48


## Load source CSV

Cell id = SWC filename stem (matches `_01`'s `index.str.removesuffix('.swc')`). Drop NaN `predicted_met_type` rows defensively (the current data has none). Validate every label is a known MET cluster id and every cell is a registered DataItem.

In [4]:
df = pd.read_csv(INPUT_CSV, index_col=0)
df.index = df.index.str.removesuffix(".swc")
print("Shape:", df.shape)
print("predicted_met_type non-null:", df["predicted_met_type"].notna().sum())
print("probability non-null       :", df["probability"].notna().sum())
df[["predicted_met_type", "probability"]].head(3)


Shape: (341, 16)
predicted_met_type non-null: 341
probability non-null       : 341


,predicted_met_type,probability
182709_6984-X2452-Y12423_reg,L5 ET-2,0.988
182709_7126-X2913-Y10535_reg,L5 ET-3,0.918
182724_5937-X3804-Y11955_reg,L5 ET-2,0.724


In [5]:
mapped_df = df.dropna(subset=["predicted_met_type"]).copy()
print(f"Cells with predicted_met_type: {len(mapped_df)} / {len(df)}")

unknown = sorted({m for m in mapped_df["predicted_met_type"].unique() if m not in met_parent})
assert not unknown, (
    f"{len(unknown)} predicted_met_type labels not in {METTYPE_HIERARCHY_ID}: {unknown[:5]}"
)

missing_cells = set(mapped_df.index) - registered_ids
assert not missing_cells, (
    f"{len(missing_cells)} CSV cells are not registered DataItems for "
    f"dataset_id='{DATASET_ID}': {sorted(missing_cells)[:5]}"
)
print(f"All {len(mapped_df)} mapped cells exist in DataItem; all labels valid.")


Cells with predicted_met_type: 341 / 341
All 341 mapped cells exist in DataItem; all labels valid.


## Write `MappingSet`

`source_dataset` is the Patch-seq cohort that defined the MET types and trained the classifier; `target_hierarchy` is the MET-types taxonomy. `target_dataset` and `source_hierarchy` left null per the `CellToClusterMapping` convention in `mappings_schema.yaml`.

In [6]:
mapping_set = MappingSet(
    id=MAPPING_SET_ID,
    name="VISp WNM excitatory MET-type assignments",
    description=(
        "Routed random forest mapping of VISp excitatory whole-neuron morphology cells "
        "onto the VISp MET-types taxonomy. Classifier trained on the Patch-seq mMET "
        "cohort (visp_exc_patchseq). Per-cell probability is recorded on the leaf row "
        "and left null on ancestor rows."
    ),
    method_name="Routed random forest mapping",
    source_dataset=MAPPING_SOURCE_DATASET_ID,
    target_hierarchy=METTYPE_HIERARCHY_ID,
    project_id=PROJECT_ID,
)

schema_ms = build_arrow_schema(MappingSet)
table_ms  = attach_linkml_metadata(
    models_to_table([mapping_set], schema=schema_ms),
    linkml_class="MappingSet",
)
write_deltalake(
    OUTPUT_ROOT + "mappingset/", table_ms,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}' AND id = '{MAPPING_SET_ID}'",
    partition_by=["project_id"],
)
print("MappingSet written:", table_ms.shape)


MappingSet written: (1, 13)


In [7]:
verify_ms = (
    pl.read_delta(OUTPUT_ROOT + "mappingset/")
      .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("id") == MAPPING_SET_ID))
)
print(verify_ms.shape)
print(verify_ms)
assert verify_ms.shape[0] == 1
assert verify_ms["source_dataset"][0]   == MAPPING_SOURCE_DATASET_ID
assert verify_ms["target_hierarchy"][0] == METTYPE_HIERARCHY_ID
assert verify_ms["target_dataset"][0]   is None
assert verify_ms["source_hierarchy"][0] is None


(1, 13)
shape: (1, 13)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ id        ┆ name      ┆ descripti ┆ method_na ┆ … ┆ source_hi ┆ target_hi ┆ json_obje ┆ project_ │
│ ---       ┆ ---       ┆ on        ┆ me        ┆   ┆ erarchy   ┆ erarchy   ┆ ct        ┆ id       │
│ str       ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ visp_exc_ ┆ VISp WNM  ┆ Routed    ┆ Routed    ┆ … ┆ null      ┆ visp_met_ ┆ null      ┆ visp_wnm │
│ wnm_metty ┆ excitator ┆ random    ┆ random    ┆   ┆           ┆ types_tax ┆           ┆          │
│ pe_mappin ┆ y         ┆ forest    ┆ forest    ┆   ┆           ┆ onomy     ┆           ┆          │
│ g         ┆ MET-type  ┆ mapping   ┆ mapping   ┆   ┆           ┆   

## Write `CellToClusterMapping`

One row per (cell, ancestor) for parent propagation up to the root. `probability` on the leaf row only; null on ancestors. Stable id: `f"{cell_id}-{cluster_id}-{PROJECT_ID}-{METTYPE_HIERARCHY_ID}"`.

In [8]:
mappings: list[CellToClusterMapping] = []
for cell_id, leaf, prob in zip(
    mapped_df.index,
    mapped_df["predicted_met_type"],
    mapped_df["probability"],
):
    leaf_prob = float(prob) if pd.notna(prob) else None
    for cid, is_leaf in walk_ancestors(leaf, met_parent):
        mappings.append(CellToClusterMapping(
            id=f"{cell_id}-{cid}-{PROJECT_ID}-{METTYPE_HIERARCHY_ID}",
            mapping_set=MAPPING_SET_ID,
            source_cell=cell_id,
            target_cluster=cid,
            probability=leaf_prob if is_leaf else None,
            project_id=PROJECT_ID,
        ))
print(f"CellToClusterMapping rows built: {len(mappings)}")

schema_ccm = build_arrow_schema(CellToClusterMapping)
table_ccm  = attach_linkml_metadata(
    models_to_table(mappings, schema=schema_ccm),
    linkml_class="CellToClusterMapping",
)
write_deltalake(
    OUTPUT_ROOT + "celltoclustermapping/", table_ccm,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}' AND mapping_set = '{MAPPING_SET_ID}'",
    partition_by=["project_id"],
)
print("CellToClusterMapping written:", table_ccm.shape)


CellToClusterMapping rows built: 1023


CellToClusterMapping written: (1023, 8)


In [9]:
verify_ccm = (
    pl.read_delta(OUTPUT_ROOT + "celltoclustermapping/")
      .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("mapping_set") == MAPPING_SET_ID))
)
print(verify_ccm.shape)
print(verify_ccm.head(3))

assert verify_ccm.shape[0] == len(mappings), (
    f"verify row count {verify_ccm.shape[0]} != built {len(mappings)}"
)
assert verify_ccm["id"].n_unique() == verify_ccm.shape[0], "duplicate CellToClusterMapping ids"
unknown = set(verify_ccm["target_cluster"].to_list()) - set(met_parent)
assert not unknown, f"target_cluster values not in {METTYPE_HIERARCHY_ID}: {sorted(unknown)[:5]}"
unknown_cells = set(verify_ccm["source_cell"].to_list()) - registered_ids
assert not unknown_cells, f"source_cell values not in DataItem: {sorted(unknown_cells)[:5]}"
assert verify_ccm["source_cell"].n_unique() == len(mapped_df), (
    f"unique source_cells {verify_ccm['source_cell'].n_unique()} != mapped cells {len(mapped_df)}"
)
# Probability is non-null exactly on leaf rows. Leaf rows = one per mapped cell.
n_with_prob = verify_ccm.filter(pl.col("probability").is_not_null()).shape[0]
assert n_with_prob == len(mapped_df), (
    f"non-null probability rows {n_with_prob} != mapped cells {len(mapped_df)}; "
    "probability should be set on leaf rows only"
)


(1023, 8)
shape: (3, 8)
┌─────────────┬─────────────┬─────────────┬─────────────┬───────┬─────────────┬───────┬────────────┐
│ id          ┆ mapping_set ┆ source_cell ┆ target_clus ┆ score ┆ probability ┆ notes ┆ project_id │
│ ---         ┆ ---         ┆ ---         ┆ ter         ┆ ---   ┆ ---         ┆ ---   ┆ ---        │
│ str         ┆ str         ┆ str         ┆ ---         ┆ f64   ┆ f64         ┆ str   ┆ str        │
│             ┆             ┆             ┆ str         ┆       ┆             ┆       ┆            │
╞═════════════╪═════════════╪═════════════╪═════════════╪═══════╪═════════════╪═══════╪════════════╡
│ 182709_6984 ┆ visp_exc_wn ┆ 182709_6984 ┆ L5 ET-2     ┆ null  ┆ 0.988       ┆ null  ┆ visp_wnm   │
│ -X2452-Y124 ┆ m_mettype_m ┆ -X2452-Y124 ┆             ┆       ┆             ┆       ┆            │
│ 23_reg-L…   ┆ apping      ┆ 23_reg      ┆             ┆       ┆             ┆       ┆            │
│ 182709_6984 ┆ visp_exc_wn ┆ 182709_6984 ┆ Glutamaterg ┆ null  ┆ n

## Summary

| Output path | Class | Rows |
|---|---|---|
| `mappingset/` (`id={MAPPING_SET_ID}`) | `MappingSet` (Routed random forest mapping) | 1 |
| `celltoclustermapping/` (`mapping_set={MAPPING_SET_ID}`) | `CellToClusterMapping` | one per (cell × MET-type ancestor); leaf rows carry `probability`, ancestors are null |

**Not written:** no `ClusterMembership` rows. WNM cells did not define the VISp MET-types taxonomy, so their assignments are mappings, not memberships.
